# Imports

In [39]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.notebook import tqdm

# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Captum
from captum.attr import (
    IntegratedGradients,
    Saliency,
    GradientShap,
    Occlusion
)

# Quantus (optional but recommended)
from quantus.metrics.faithfulness.faithfulness_correlation import FaithfulnessCorrelation
from quantus.metrics.complexity.complexity import Complexity
from quantus.metrics.complexity.sparseness import Sparseness

# Stats
from scipy.stats import spearmanr

# Visualization (optional)
import seaborn as sns
import matplotlib.pyplot as plt

In [40]:
# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

# ============================================================
# Dataset Loading
# ============================================================

BASE_DIR = "TFE_Data"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")

df = pd.read_pickle(os.path.join(DATASETS_DIR, "df_Flickr8k.pkl"))
IMAGE_PATHS = df["image_path"].tolist()

# Limit to 50 images for benchmarking consistency
MAX_IMAGES = 50
IMAGE_PATHS = IMAGE_PATHS[:MAX_IMAGES]

Using device: cuda


In [41]:
def load_vision_embeddings(path):
    """
    Loads precomputed unimodal vision embeddings.
    Shape: (N_images, D)
    """
    return torch.tensor(np.load(path), dtype=torch.float32).to(device)

def load_text_embeddings(path):
    """
    Loads precomputed unimodal text embeddings.
    Shape: (N_captions, D)
    """
    return torch.tensor(np.load(path), dtype=torch.float32).to(device)

# Model Wrappers

In [42]:
class ProjectionModel(nn.Module):
    def __init__(self, W):
        super().__init__()
        self.W = nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False)

    def forward(self, x):
        y = x @ self.W          # (batch, dim)
        return y.norm(dim=1, keepdim=True)  # <-- scalar output per sample


In [43]:
class CLIPProjection(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model

    def forward(self, x):
        return self.clip.encode_image(x)


# Attribution

In [44]:
def explain_ig(model, emb, target=None):
    ig = IntegratedGradients(model)
    return ig.attribute(emb.unsqueeze(0), target=target)

def explain_saliency(model, emb, target=None):
    sal = Saliency(model)
    return sal.attribute(emb.unsqueeze(0), target=target)

def explain_gs(model, emb, target=None):
    gs = GradientShap(model)
    baseline = torch.zeros_like(emb)
    return gs.attribute(emb.unsqueeze(0), baselines=baseline.unsqueeze(0), target=target)

def explain_occlusion(model, emb, target=None):
    occ = Occlusion(model)
    return occ.attribute(
        emb.unsqueeze(0),
        target=target,
        sliding_window_shapes=(1, 32),
        strides=(1, 16)
    )


# Explainability Metrics

In [45]:
def faithfulness_test(model, emb, attr):
    with torch.no_grad():
        base = model(emb.unsqueeze(0)).norm().item()

        # mask top-k attribution dims
        a = attr.squeeze().abs()
        k = int(0.1 * len(a))
        topk = torch.topk(a, k).indices

        emb_mod = emb.clone()
        emb_mod[topk] = 0

        new = model(emb_mod.unsqueeze(0)).norm().item()

    return base - new


In [46]:
def sparsity(attr):
    a = attr.squeeze().abs().cpu().numpy()
    return (np.sum(a < 1e-5) / len(a))


In [47]:
def complexity(attr):
    a = attr.squeeze().abs().cpu().numpy()
    return np.std(a)


In [48]:
def rank_corr(attr1, attr2):
    a1 = attr1.squeeze().cpu().numpy()
    a2 = attr2.squeeze().cpu().numpy()
    return spearmanr(a1, a2).correlation


# Multimodal Alignment Analysis

In [49]:
def cluster_consistency(attrs):
    sims = []
    for i in range(len(attrs)):
        for j in range(i+1, len(attrs)):
            a1 = attrs[i].squeeze().cpu().numpy()
            a2 = attrs[j].squeeze().cpu().numpy()
            sim = np.dot(a1,a2)/(np.linalg.norm(a1)*np.linalg.norm(a2)+1e-8)
            sims.append(sim)
    return np.mean(sims)


In [50]:
def cluster_drift(cluster_centroids):
    C = len(cluster_centroids)
    drift = np.zeros((C,C))
    for i in range(C):
        for j in range(C):
            a1 = cluster_centroids[i]
            a2 = cluster_centroids[j]
            drift[i,j] = 1 - np.dot(a1,a2)/(np.linalg.norm(a1)*np.linalg.norm(a2)+1e-8)
    return drift


In [51]:
def caption_similarity_distribution(img_embs, cap_embs):
    sims = []
    for i in range(len(img_embs)):
        for j in range(5):  # 5 captions
            sims.append(
                F.cosine_similarity(
                    img_embs[i].unsqueeze(0),
                    cap_embs[i*5+j].unsqueeze(0)
                ).item()
            )
    return sims


# Pipeline

In [52]:
def benchmark_projection(name, proj_model, vision_embs, text_embs, clusters):
    results = []

    for i, emb in enumerate(vision_embs):
        # 1. Compute IG
        ig = explain_ig(proj_model, emb)

        # 2. Compute metrics
        res = {
            "model": name,
            "image_idx": i,
            "faithfulness": faithfulness_test(proj_model, emb, ig),
            "sparsity": sparsity(ig),
            "complexity": complexity(ig),
        }

        # rank corr with saliency
        sal = explain_saliency(proj_model, emb)
        res["rank_corr"] = rank_corr(ig, sal)

        results.append(res)

    # 3. Cluster-level metrics
    cluster_attrs = {c: [] for c in np.unique(clusters)}
    for i, c in enumerate(clusters):
        ig = explain_ig(proj_model, vision_embs[i])
        cluster_attrs[c].append(ig)

    # consistency
    for c in cluster_attrs:
        results.append({
            "model": name,
            "cluster": c,
            "consistency": cluster_consistency(cluster_attrs[c])
        })

    # drift
    centroids = [
        np.mean([a.squeeze().cpu().numpy() for a in cluster_attrs[c]], axis=0)
        for c in cluster_attrs
    ]
    drift = cluster_drift(centroids)

    return results, drift


In [53]:
def summarize(results):
    df = pd.DataFrame(results)
    return df.groupby("model").mean(numeric_only=True)


# Execution

In [69]:
import glob
import re

def discover_projection_configs(base_dir):
    """
    Automatically finds all (vision, text, projection) combinations
    by scanning filenames.
    """
    configs = []

    # Example pattern: Wt_mobilenet_v3_bert_cpca.npy
    pattern = re.compile(r"Wt_(.+)_(.+)_(.+)\.npy")

    for wt_path in glob.glob(os.path.join(base_dir, "projection_matrices", "Wt_*.npy")):
        filename = os.path.basename(wt_path)
        match = pattern.match(filename)
        if not match:
            continue

        vision, text, proj = match.groups()

        config = {
            "vision": vision,
            "text": text,
            "proj": proj,
            "Wt": wt_path,
            "Wv": wt_path.replace("Wt_", "Wv_"),
            "Xv_proj": f"/home/aysel/tfe/projected_embeddings/{vision}_{text}_{proj}_Xv.npy",
            "Xt_proj": f"/home/aysel/tfe/projected_embeddings/{vision}_{text}_{proj}_Xt.npy",
            "vision_emb": f"/home/aysel/tfe/TFE_Data/Unimodal_Results/Flickr8k/vision/{vision}/embeddings.npy",
            "text_emb": f"/home/aysel/tfe/TFE_Data/Unimodal_Results/Flickr8k/text/{text}/embeddings.npy",
        }

        configs.append(config)

    return configs

def load_projection_model(config):
    Wv = np.load(config["Wv"])
    return ProjectionModel(Wv).to(device).eval()
def load_embeddings(config):
    vision_embs = torch.tensor(np.load(config["vision_emb"]), dtype=torch.float32).to(device)
    text_embs   = torch.tensor(np.load(config["text_emb"]), dtype=torch.float32).to(device)
    return vision_embs, text_embs


In [70]:
configs = discover_projection_configs("/home/aysel/tfe")
print("Found", len(configs), "projection configurations")


Found 12 projection configurations


In [56]:
VALID_PROJ = {"wcca", "cpca", "random"}

configs = [
    cfg for cfg in discover_projection_configs("/home/aysel/tfe")
    if cfg["proj"].lower() in VALID_PROJ
]

print("Valid configs:", len(configs))
for c in configs:
    print(c["vision"], c["text"], c["proj"])


Valid configs: 12
mobilenet_v3 bert wcca
pvt bert cpca
mobilenet_v3 roberta random
mobilenet_v3 roberta cpca
pvt roberta wcca
pvt roberta cpca
mobilenet_v3 bert random
pvt roberta random
mobilenet_v3 roberta wcca
pvt bert random
mobilenet_v3 bert cpca
pvt bert wcca


In [57]:
from sklearn.cluster import KMeans


In [58]:
all_results = []
all_drifts = {}

for cfg in configs:
    name = f"{cfg['vision']}_{cfg['text']}_{cfg['proj']}"
    print("Running:", name)

    proj_model = load_projection_model(cfg)
    vision_embs, text_embs = load_embeddings(cfg)

    # cluster the projected vision embeddings
    clusters = KMeans(n_clusters=10).fit_predict(vision_embs.cpu().numpy())

    results, drift = benchmark_projection(
        name=name,
        proj_model=proj_model,
        vision_embs=vision_embs,
        text_embs=text_embs,
        clusters=clusters
    )

    all_results.extend(results)
    all_drifts[name] = drift


Running: mobilenet_v3_bert_wcca


/home/aysel/tfe/.venv/lib/python3.12/site-packages/captum/attr/_core/saliency.py:129: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


Running: pvt_bert_cpca


/home/aysel/tfe/.venv/lib/python3.12/site-packages/captum/attr/_core/saliency.py:129: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


Running: mobilenet_v3_roberta_random


...

/home/aysel/tfe/.venv/lib/python3.12/site-packages/captum/attr/_core/saliency.py:129: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


Running: mobilenet_v3_bert_cpca


/home/aysel/tfe/.venv/lib/python3.12/site-packages/captum/attr/_core/saliency.py:129: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


Running: pvt_bert_wcca


/home/aysel/tfe/.venv/lib/python3.12/site-packages/captum/attr/_core/saliency.py:129: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


In [62]:
df_results = pd.DataFrame(all_results)

df_results_clean = df_results.drop(columns=["image_idx", "cluster"])
summary = df_results_clean.groupby("model").mean(numeric_only=True)

summary


,faithfulness,sparsity,complexity,rank_corr,consistency
model,,,,,
mobilenet_v3_bert_cpca,9.652515,0.063380,0.065343,0.676709,0.306222
mobilenet_v3_bert_random,95.864075,0.021630,0.920561,0.181845,0.258685
mobilenet_v3_bert_wcca,9.432940,0.059174,0.063858,0.652054,0.335782
mobilenet_v3_roberta_cpca,9.646768,0.063394,0.065324,0.677294,0.322756
mobilenet_v3_roberta_random,95.450184,0.021641,0.933388,0.185577,0.237417
mobilenet_v3_roberta_wcca,9.429834,0.059311,0.063838,0.651484,0.311457
pvt_bert_cpca,7.607242,0.003445,0.067032,0.798916,0.370073
pvt_bert_random,96.308245,0.000081,1.076444,0.408691,0.198207
pvt_bert_wcca,7.278881,0.003080,0.063804,0.757459,0.359908


# Added Multimodal Retrieval

In [63]:
def load_projection_models(config):
    Wv = np.load(config["Wv"])
    Wt = np.load(config["Wt"])
    proj_v = ProjectionModel(Wv).to(device).eval()
    proj_t = ProjectionModel(Wt).to(device).eval()
    return proj_v, proj_t

def load_projected_embeddings(config):
    Xv = torch.tensor(np.load(config["Xv_proj"]), dtype=torch.float32).to(device)
    Xt = torch.tensor(np.load(config["Xt_proj"]), dtype=torch.float32).to(device)
    return Xv, Xt


In [64]:
def cross_modal_consistency(ig_img, ig_txt):
    a = ig_img.squeeze().cpu().numpy()
    b = ig_txt.squeeze().cpu().numpy()
    return np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-8)


In [65]:
def attribution_drift(a, b):
    a = a.squeeze().cpu().numpy()
    b = b.squeeze().cpu().numpy()
    return 1 - np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-8)


In [66]:
def caption_similarity_distribution(img_embs, cap_embs):
    sims = []
    for i in range(len(img_embs)):
        for j in range(5):
            sims.append(F.cosine_similarity(
                img_embs[i].unsqueeze(0),
                cap_embs[i*5+j].unsqueeze(0)
            ).item())
    return sims


In [74]:
all_results = []
all_drift = []

for cfg in configs:

    name = f"{cfg['vision']}_{cfg['text']}_{cfg['proj']}"
    print("Running:", name)

    # ----------------------------------------------------
    # Load unimodal embeddings for THIS config
    # ----------------------------------------------------
    vision_embs = torch.tensor(np.load(cfg["vision_emb"]), dtype=torch.float32).to(device)
    text_embs   = torch.tensor(np.load(cfg["text_emb"]),   dtype=torch.float32).to(device)

    # ----------------------------------------------------
    # Load projection matrices
    # ----------------------------------------------------
    Wv = np.load(cfg["Wv"])
    Wt = np.load(cfg["Wt"])

    # Dimension check
    if vision_embs.shape[1] != Wv.shape[0]:
        print(" → Skipped: vision embedding dim mismatch")
        continue

    if text_embs.shape[1] != Wt.shape[0]:
        print(" → Skipped: text embedding dim mismatch")
        continue

    # Build projection models
    proj_v = ProjectionModel(Wv).to(device).eval()
    proj_t = ProjectionModel(Wt).to(device).eval()

    # ----------------------------------------------------
    # Load projected embeddings (for similarity only)
    # ----------------------------------------------------
    if not (os.path.exists(cfg["Xv_proj"]) and os.path.exists(cfg["Xt_proj"])):
        print(" → Skipped: projected embeddings missing")
        continue

    Xv = torch.tensor(np.load(cfg["Xv_proj"]), dtype=torch.float32).to(device)
    Xt = torch.tensor(np.load(cfg["Xt_proj"]), dtype=torch.float32).to(device)

    # ----------------------------------------------------
    # 1. CROSS-MODAL CONSISTENCY
    # ----------------------------------------------------
    cm_consist = []
    for i in range(len(vision_embs)):
        # IG in input space
        ig_img = explain_ig(proj_v, vision_embs[i]).cpu().numpy()      # shape (D_v,)
        ig_txt = explain_ig(proj_t, text_embs[i*5]).cpu().numpy()      # shape (D_t,)

        # Project IG into shared space
        ig_img_proj = ig_img @ Wv      # shape (128,)
        ig_txt_proj = ig_txt @ Wt      # shape (128,)

        # Convert back to tensor for consistency
        ig_img_proj = torch.tensor(ig_img_proj, dtype=torch.float32)
        ig_txt_proj = torch.tensor(ig_txt_proj, dtype=torch.float32)

        # Now compare
        cm_consist.append(cross_modal_consistency(ig_img_proj, ig_txt_proj))


    # ----------------------------------------------------
    # 2. CAPTION SIMILARITY
    # ----------------------------------------------------
    sims = caption_similarity_distribution(Xv, Xt)

    # ----------------------------------------------------
    # 3. ATTRIBUTION DRIFT (store IGs on UNIMODAL embeddings)
    # ----------------------------------------------------
    igs_for_drift = [explain_ig(proj_v, vision_embs[i]).cpu().numpy()
                     for i in range(len(vision_embs))]

    all_drift.append({
        "model": name,
        "ig_img": igs_for_drift
    })

    # ----------------------------------------------------
    # STORE RESULTS
    # ----------------------------------------------------
    all_results.append({
        "model": name,
        "cross_modal_consistency": np.mean(cm_consist),
        "mean_similarity": np.mean(sims),
        "std_similarity": np.std(sims)
    })


Running: mobilenet_v3_bert_wcca
Running: pvt_bert_cpca
Running: mobilenet_v3_roberta_random
Running: mobilenet_v3_roberta_cpca
Running: pvt_roberta_wcca
Running: pvt_roberta_cpca
Running: mobilenet_v3_bert_random
Running: pvt_roberta_random
Running: mobilenet_v3_roberta_wcca
Running: pvt_bert_random
Running: mobilenet_v3_bert_cpca
Running: pvt_bert_wcca


In [77]:
import os

os.makedirs("results_multimodal/metrics", exist_ok=True)
os.makedirs("results_multimodal/drift", exist_ok=True)
os.makedirs("results_multimodal/bundle", exist_ok=True)

# Convert to DataFrames
df_multi = pd.DataFrame(all_results)

# Save metrics
df_multi.to_csv("results_multimodal/metrics/multimodal_metrics.csv", index=False)
with open("results_multimodal/metrics/multimodal_metrics.pkl", "wb") as f:
    pickle.dump(all_results, f)

# Save drift vectors
with open("results_multimodal/drift/multimodal_ig_drift.pkl", "wb") as f:
    pickle.dump(all_drift, f)

# Save drift table
df_drift = pd.DataFrame(drift_rows)
df_drift.to_csv("results_multimodal/drift/multimodal_drift.csv", index=False)
with open("results_multimodal/drift/multimodal_drift.pkl", "wb") as f:
    pickle.dump(df_drift, f)

# Save everything together
bundle = {
    "results": all_results,
    "drift_vectors": all_drift,
    "drift_table": drift_rows
}
with open("results_multimodal/bundle/multimodal_explainability_bundle.pkl", "wb") as f:
    pickle.dump(bundle, f)

print("All multimodal explainability results saved successfully.")


All multimodal explainability results saved successfully.


In [75]:
drift_rows = []

for i in range(len(all_drift)):
    for j in range(i+1, len(all_drift)):

        m1 = all_drift[i]["model"]
        m2 = all_drift[j]["model"]

        ig1 = all_drift[i]["ig_img"]
        ig2 = all_drift[j]["ig_img"]

        drifts = [
            attribution_drift(ig1[k], ig2[k])
            for k in range(len(ig1))
        ]

        drift_rows.append({
            "model_pair": f"{m1}__{m2}",
            "drift": np.mean(drifts)
        })

df_drift = pd.DataFrame(drift_rows)


AttributeError: 'numpy.ndarray' object has no attribute 'cpu'